# 🎯 Notebook 09: Hyperparameter Tuning
## Customer Churn Prediction
- GridSearchCV for Top 2 Models
- RandomizedSearchCV for XGBoost
- Scoring: Recall

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from pathlib import Path
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
print("✅ Libraries imported!")

if os.path.exists('/kaggle/working'):
    X_train = np.load('/kaggle/working/X_train.npy')
    y_train = np.load('/kaggle/working/y_train.npy')
    MODEL_DIR = Path('/kaggle/working/models')
else:
    X_train = np.load('../models/X_train.npy')
    y_train = np.load('../models/y_train.npy')
    MODEL_DIR = Path('../models')

In [ ]:
print("🔍 Tuning Random Forest...")
rf_param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), rf_param_grid, cv=5, scoring='recall', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)
print(f"✅ Best RF Params: {rf_grid.best_params_}")
print(f"✅ Best RF Recall: {rf_grid.best_score_:.4f}")

In [ ]:
print("🔍 Tuning XGBoost...")
xgb_param_dist = {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.2], 'subsample': [0.8, 1.0]}
xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1), xgb_param_dist, n_iter=10, cv=5, scoring='recall', n_jobs=-1, verbose=1, random_state=42)
xgb_random.fit(X_train, y_train)
print(f"✅ Best XGB Params: {xgb_random.best_params_}")
print(f"✅ Best XGB Recall: {xgb_random.best_score_:.4f}")

In [ ]:
if rf_grid.best_score_ > xgb_random.best_score_:
    best_tuned = rf_grid.best_estimator_
    print("🏆 Random Forest is the winner!")
else:
    best_tuned = xgb_random.best_estimator_
    print("🏆 XGBoost is the winner!")
joblib.dump(best_tuned, MODEL_DIR / 'best_model.pkl')
print(f"💾 Saved tuned best model to: {MODEL_DIR / 'best_model.pkl'}")
print("✅ Notebook 09 Complete!")